# MALTO "Who Moves?" - final submission notebook (final_stage6_edge)

Reproduces our final submission from the competition data, top to bottom.

**Pipeline**
1. **Detector fine-tuning.** YOLO11x (Ultralytics, COCO-pretrained weights `yolo11x.pt`, AGPL-3.0) is fine-tuned on frames of
   the training videos (one frame every 2 s), with the competition's training boxes as labels (person -> COCO person,
   vehicle -> COCO car, animal -> COCO dog; `disturbance` and unlabelled `other` boxes dropped). Training scenes are
   parking lot A and the other scenes; parking lot B is kept out and used to pick the best epoch (scene-held-out validation).
   First 10 layers frozen, AdamW, lr 0.001, image size 1280, 23 epochs, fixed seed, deterministic mode.
2. **Detection** on every frame of every test video (10 fps): COCO YOLO11x on the whole frame (1280 px) and on 2x2
   overlapping tiles (960 px each), plus the fine-tuned YOLO11x on the whole frame, of which only **person** boxes with
   score >= 0.25 are used (the fine-tuned model finds small people the COCO model misses). All boxes are merged with
   same-class NMS (IoU 0.5).
3. **Tracking** - our own tracker: constant-velocity Kalman filter (DeepSORT-style state), Hungarian matching on
   1 - IoU + normalised centre distance, ByteTrack-style second pass with low-confidence boxes, class-group NMS.
4. **Decision** - one alarm per track, at the first moment the track is at least 1 s old and has moved more than one box
   height from where it started (free box edges used for boxes touching the border). A box touching the border must be at
   least 20 px tall to raise an alarm.

No VLM / text-driven model, no test-video inspection or labelling, no source-dataset annotations. Every threshold was
chosen on the training set with scene-grouped evaluation.

**Settings:** GPU T4 x2, Internet on, competition data attached. Runtime ~5 h.
**Outputs:** `submission.csv` (the final submission), `best.pt`, detections.

In [1]:
# 1) Setup
import glob, os, shutil, subprocess, sys, time
import numpy as np, pandas as pd, cv2
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    import ultralytics
import torch
from ultralytics import YOLO
DATA = os.path.dirname(glob.glob("/kaggle/input/**/train_videos.csv", recursive=True)[0])
OUT = "/kaggle/working"; FT = f"{OUT}/ft"
VIDS = pd.read_csv(f"{DATA}/train_videos.csv", dtype={"video_id": str}).set_index("video_id")
TEST = pd.read_csv(f"{DATA}/test_videos.csv", dtype={"video_id": str}).set_index("video_id")
COCO = YOLO("yolo11x.pt"); NAMES = COCO.names; COCO_W = os.path.abspath("yolo11x.pt")
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPUs", torch.cuda.device_count())
print("data:", DATA, "|", len(VIDS), "train videos,", len(TEST), "test videos")
# scene groups of the training videos (by parked-car layout, computed from our own detections): A, B, other
GROUP = {"train_0001e6b8": "B", "train_04c8087d": "B", "train_087daea2": "B", "train_09b158a0": "A", "train_0e7144b1": "O", "train_0eadc735": "A", "train_0f9697f8": "B", "train_17bddc9f": "A", "train_1a6d556f": "B", "train_1bf7076b": "B", "train_1d66c570": "B", "train_1e5ebced": "A", "train_262a0d98": "B", "train_298a12e2": "B", "train_299a7d52": "A", "train_323cd259": "A", "train_32cef231": "B", "train_335d9fce": "A", "train_38ab071f": "A", "train_394f5d71": "A", "train_3a6fa6b9": "A", "train_3e19cfb9": "A", "train_3ecf24bc": "B", "train_44a67e0a": "A", "train_45d2a81f": "B", "train_48d2d491": "B", "train_48da9eaa": "A", "train_4a11cf07": "B", "train_507665b0": "B", "train_50d3afc0": "A", "train_513ef966": "A", "train_51e8f621": "A", "train_57acee2b": "A", "train_5c944206": "A", "train_67c68dad": "B", "train_692dbc01": "A", "train_6be8bd6d": "B", "train_728f0655": "B", "train_72d7e66c": "A", "train_7b81a79f": "B", "train_82fdac3a": "A", "train_85290c5e": "A", "train_8805fd7d": "B", "train_8b2e7ebd": "A", "train_8b72e875": "O", "train_8dc74776": "O", "train_95366bf2": "A", "train_966d2c49": "A", "train_96dc44ce": "A", "train_9b741ef1": "B", "train_a1669c8b": "A", "train_a580cb16": "B", "train_a6dfba9e": "O", "train_a78dc858": "A", "train_ac128460": "B", "train_aded762a": "O", "train_ae5db495": "A", "train_ae931833": "B", "train_b23c2b1b": "A", "train_b2f332a1": "A", "train_b60956c2": "A", "train_b8c34e8b": "O", "train_b8f99f37": "A", "train_bd0dd247": "B", "train_bfb55789": "A", "train_c1c6f2ab": "B", "train_c717ae39": "B", "train_c74d48c1": "B", "train_c9f88c5d": "A", "train_cc59cbba": "A", "train_ccb1e5c4": "A", "train_d4c337b4": "A", "train_d998b0e4": "A", "train_dca31900": "O", "train_e11ed304": "A", "train_e216014c": "A", "train_e3cabd9f": "B", "train_e4d4652f": "B", "train_e59c8b00": "B", "train_e73b5b8a": "A", "train_ea4a3c24": "B", "train_ec0660f1": "O", "train_edbd6f80": "A", "train_edfaac2a": "O", "train_f090f52a": "A", "train_f1fcca7e": "A", "train_f915c640": "B", "train_fa0ec47b": "A", "train_fe39f21f": "B", "train_ff27c540": "A"}

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ultralytics 8.4.188 | torch 2.11.0+cu128 | GPUs 2
data: /kaggle/input/competitions/malto-who-moves | 90 train videos, 48 test videos


In [2]:
# 2) Training labels and frames: train = lot A + other (1 frame / 2 s), val = lot B (1 frame / 10 s)
L = pd.read_csv(f"{DATA}/train_tracks.csv", dtype={"video_id": str})
L = L[(L.role != "disturbance") & L["class"].isin(["person", "vehicle", "animal"])]
CLS = {"person": 0, "vehicle": 2, "animal": 16}
def vpath(v):
    q = f"{DATA}/{v}.mp4"
    return q if os.path.exists(q) else glob.glob(f"{DATA}/**/{v}.mp4", recursive=True)[0]
stats = {"train": 0, "val": 0}
G = {v: g for v, g in L.groupby("video_id")}
for v in VIDS.index:
    split, step = ("val", 100) if GROUP[v] == "B" else ("train", 20)
    os.makedirs(f"{FT}/images/{split}", exist_ok=True); os.makedirs(f"{FT}/labels/{split}", exist_ok=True)
    lab = G.get(v, L.iloc[:0]); byf = {f: g for f, g in lab.groupby("frame")}
    cap = cv2.VideoCapture(vpath(v)); W = int(cap.get(3)); H = int(cap.get(4)); fi = 0
    while True:
        ok, img = cap.read()
        if not ok: break
        if fi % step == 0:
            name = f"{v}_{fi:05d}"
            cv2.imwrite(f"{FT}/images/{split}/{name}.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            g = byf.get(fi); lines = []
            if g is not None:
                x1 = g.x.clip(0, W); y1 = g.y.clip(0, H); x2 = (g.x + g.w).clip(0, W); y2 = (g.y + g.h).clip(0, H)
                for c, a, b, cc, d in zip(g["class"], x1, y1, x2, y2):
                    if cc - a < 2 or d - b < 2: continue
                    lines.append(f"{CLS[c]} {(a + cc) / 2 / W:.6f} {(b + d) / 2 / H:.6f} {(cc - a) / W:.6f} {(d - b) / H:.6f}")
            open(f"{FT}/labels/{split}/{name}.txt", "w").write("\n".join(lines))
            stats[split] += 1
        fi += 1
    cap.release()
print("images:", stats)
open(f"{FT}/data.yaml", "w").write(f"path: {FT}\ntrain: images/train\nval: images/val\nnames:\n" + "".join(f"  {k}: {n}\n" for k, n in NAMES.items()))

images: {'train': 4390, 'val': 450}


1159

In [3]:
# 3) Fine-tune YOLO11x (both GPUs, DDP through the CLI). The best epoch on lot B is kept (best.pt).
t0 = time.time()
cmd = ["yolo", "detect", "train", "model=yolo11x.pt", f"data={FT}/data.yaml", "imgsz=1280", "epochs=23",
       "batch=8", "device=0,1", "workers=4", "freeze=10", "optimizer=AdamW", "lr0=0.001", "cos_lr=True",
       "warmup_epochs=1", "mosaic=0.3", "close_mosaic=3", "scale=0.3", "degrees=0", "fliplr=0.5",
       "plots=False", f"project={OUT}/runs", "name=ft", "exist_ok=True", "seed=0", "deterministic=True"]
logf = open(f"{OUT}/train_log.txt", "w")
p = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
while p.poll() is None:
    time.sleep(300)
    res = f"{OUT}/runs/ft/results.csv"
    if os.path.exists(res):
        r = pd.read_csv(res); r.columns = [c.strip() for c in r.columns]; last = r.iloc[-1]
        print(f"[{(time.time()-t0)/60:5.0f} min] epoch {int(last['epoch'])}/23  lot B mAP50 {last['metrics/mAP50(B)']:.3f}  mAP50-95 {last['metrics/mAP50-95(B)']:.3f}", flush=True)
logf.close()
assert p.returncode == 0, open(f"{OUT}/train_log.txt").read()[-4000:]
FT_W = f"{OUT}/best.pt"; shutil.copy(f"{OUT}/runs/ft/weights/best.pt", FT_W); shutil.copy(f"{OUT}/runs/ft/results.csv", f"{OUT}/results.csv")
shutil.rmtree(FT, ignore_errors=True)
print(f"fine-tuning done in {(time.time()-t0)/60:.0f} min")

[   10 min] epoch 1/23  lot B mAP50 0.469  mAP50-95 0.296
[   15 min] epoch 2/23  lot B mAP50 0.483  mAP50-95 0.312
[   20 min] epoch 2/23  lot B mAP50 0.483  mAP50-95 0.312
[   25 min] epoch 3/23  lot B mAP50 0.468  mAP50-95 0.300
[   30 min] epoch 4/23  lot B mAP50 0.485  mAP50-95 0.307
[   35 min] epoch 5/23  lot B mAP50 0.499  mAP50-95 0.327
[   40 min] epoch 5/23  lot B mAP50 0.499  mAP50-95 0.327
[   45 min] epoch 6/23  lot B mAP50 0.486  mAP50-95 0.304
[   50 min] epoch 7/23  lot B mAP50 0.487  mAP50-95 0.325
[   55 min] epoch 8/23  lot B mAP50 0.488  mAP50-95 0.308
[   60 min] epoch 8/23  lot B mAP50 0.488  mAP50-95 0.308
[   65 min] epoch 9/23  lot B mAP50 0.489  mAP50-95 0.329
[   70 min] epoch 10/23  lot B mAP50 0.490  mAP50-95 0.306
[   75 min] epoch 11/23  lot B mAP50 0.489  mAP50-95 0.314
[   80 min] epoch 11/23  lot B mAP50 0.489  mAP50-95 0.314
[   85 min] epoch 12/23  lot B mAP50 0.496  mAP50-95 0.307
[   90 min] epoch 13/23  lot B mAP50 0.488  mAP50-95 0.304
[   95 mi

In [4]:
# 4) Detection on the test videos: GPU 0 = COCO tiles; GPU 1 = COCO 1280 then fine-tuned 1280
WORKER = r"""
import glob, os, sys, time, logging
import numpy as np, pandas as pd, cv2
from ultralytics import YOLO
from ultralytics.utils import LOGGER
LOGGER.setLevel(logging.ERROR)
DATA, OUT, jobs, videos = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4].split(",")
SCORE_MIN, OVL, BATCH = 0.10, 64, 8
KEEP = {"person", "bicycle", "car", "motorcycle", "bus", "train", "truck", "bird",
        "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe"}
def path(vid):
    p = f"{DATA}/{vid}.mp4"
    return p if os.path.exists(p) else glob.glob(f"{DATA}/**/{vid}.mp4", recursive=True)[0]
def tiles(W, H):
    tw, th = (W + 1) // 2 + OVL, (H + 1) // 2 + OVL
    return [(x, y, min(x + tw, W), min(y + th, H)) for y in (0, H - th) for x in (0, W - tw)]
for job in jobs.split(";"):
    tag, mode, weights = job.split("|")
    model = YOLO(weights); names = model.names
    def predict(imgs, imgsz):
        res = model.predict(imgs, imgsz=imgsz, conf=SCORE_MIN, iou=0.7, half=True, device=0, verbose=False)
        return [(r.boxes.xyxy.float().cpu().numpy(), r.boxes.conf.float().cpu().numpy(),
                 [names[int(c)] for c in r.boxes.cls.cpu()]) for r in res]
    rows = []
    for k, vid in enumerate(videos):
        cap = cv2.VideoCapture(path(vid)); W = int(cap.get(3)); H = int(cap.get(4))
        T = tiles(W, H); buf = []; fi = 0; t0 = time.time()
        def flush():
            if mode == "1280":
                for (i, _), (b, s, l) in zip(buf, predict([f for _, f in buf], 1280)):
                    for (x1, y1, x2, y2), sc, lab in zip(b, s, l):
                        if lab in KEEP: rows.append((vid, i, x1, y1, x2, y2, sc, lab))
            else:
                crops = [f[y1:y2, x1:x2] for _, f in buf for (x1, y1, x2, y2) in T]
                for n, (b, s, l) in enumerate(predict(crops, 960)):
                    i = buf[n // len(T)][0]; ox, oy, ex, ey = T[n % len(T)]
                    for (x1, y1, x2, y2), sc, lab in zip(b, s, l):
                        if lab not in KEEP: continue
                        X1, Y1, X2, Y2 = x1 + ox, y1 + oy, x2 + ox, y2 + oy
                        if (ox > 0 and X1 <= ox + 2) or (oy > 0 and Y1 <= oy + 2) or (ex < W and X2 >= ex - 2) or (ey < H and Y2 >= ey - 2):
                            continue
                        rows.append((vid, i, X1, Y1, X2, Y2, sc, lab))
            buf.clear()
        while True:
            ok, f = cap.read()
            if not ok: break
            buf.append((fi, f)); fi += 1
            if len(buf) == BATCH: flush()
        if buf: flush()
        cap.release()
        print(f"[{tag} {k+1}/{len(videos)}] {vid}: {fi} frames in {time.time()-t0:.0f} s", flush=True)
    df = pd.DataFrame(rows, columns=["video_id", "frame", "x1", "y1", "x2", "y2", "score", "label"])
    df[["x1", "y1", "x2", "y2"]] = df[["x1", "y1", "x2", "y2"]].astype(float).round(1); df["score"] = df.score.astype(float).round(3)
    df.to_csv(f"{OUT}/dets_test_{tag}.csv.gz", index=False, compression="gzip")
    print("saved", tag, len(df), "boxes", flush=True)
"""
open(f"{OUT}/worker.py", "w").write(WORKER)
vids = ",".join(TEST.index)
plan = [(0, "cocotile", f"cocotile|tile|{COCO_W}"), (1, "coco1280+ft1280", f"coco1280|1280|{COCO_W};ft1280|1280|{FT_W}")]
t0 = time.time(); procs = []
for gpu, name, jobs in plan:
    logf = open(f"{OUT}/stdout_{gpu}.txt", "w")
    procs.append((name, subprocess.Popen([sys.executable, f"{OUT}/worker.py", DATA, OUT, jobs, vids],
                 env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu % max(torch.cuda.device_count(), 1)), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"),
                 stdout=logf, stderr=subprocess.STDOUT), logf, gpu))
while any(p.poll() is None for _, p, _, _ in procs):
    time.sleep(300)
    for name, p, _, gpu in procs:
        lines = open(f"{OUT}/stdout_{gpu}.txt").read().strip().splitlines()
        print(f"[{(time.time()-t0)/60:5.0f} min] {name}: {lines[-1] if lines else 'starting'}", flush=True)
for name, p, logf, gpu in procs:
    logf.close(); assert p.returncode == 0, open(f"{OUT}/stdout_{gpu}.txt").read()[-3000:]
print(f"detection done in {(time.time()-t0)/60:.0f} min")

[    5 min] cocotile: [cocotile 2/48] 19c04643: 600 frames in 120 s
[    5 min] coco1280+ft1280: [coco1280 7/48] 2b51f738: 600 frames in 38 s
[   10 min] cocotile: [cocotile 5/48] 25d91ca9: 600 frames in 95 s
[   10 min] coco1280+ft1280: [coco1280 13/48] 487e3d59: 1200 frames in 74 s
[   15 min] cocotile: [cocotile 8/48] 3170d37f: 600 frames in 95 s
[   15 min] coco1280+ft1280: [coco1280 18/48] 6edb3b8c: 571 frames in 37 s
[   20 min] cocotile: [cocotile 11/48] 3797ef30: 439 frames in 71 s
[   20 min] coco1280+ft1280: [coco1280 24/48] 8d81f590: 600 frames in 38 s
[   25 min] cocotile: [cocotile 13/48] 487e3d59: 1200 frames in 188 s
[   25 min] coco1280+ft1280: [coco1280 29/48] 9d3a8d74: 1200 frames in 90 s
[   30 min] cocotile: [cocotile 14/48] 4b815206: 900 frames in 163 s
[   30 min] coco1280+ft1280: [coco1280 35/48] b9304a42: 535 frames in 34 s
[   35 min] cocotile: [cocotile 17/48] 58f06b4c: 600 frames in 96 s
[   35 min] coco1280+ft1280: [coco1280 40/48] d5ce9e78: 600 frames in 50

In [5]:
# 5) Decision pipeline (identical to malto_pipeline.py in our repository)
"""MALTO "Who Moves?" - complete decision pipeline in one file (no project imports).

detector boxes (CSV: video_id, frame, x1, y1, x2, y2, score, label)
  -> class mapping (person / vehicle / animal), merge of several detector runs (same-class NMS)
  -> tracker: Kalman filter + Hungarian matching, ByteTrack-style two-stage association, class-group NMS
  -> optional ID-switch split: cut a track where its box jumps > 0.8 box heights in one frame
  -> per-observation motion features computed only from the past
  -> rule: first moment with age >= 1 s and displacement > 1 box height (edge-aware) -> one alarm per track
  -> submission CSV (video_id, alarms)
Dependencies: numpy, pandas, scipy.
"""
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment


# ---------------------------------------------------------------- classes and geometry

VEH = {'motorcycle', 'train', 'car', 'truck', 'motorbike', 'bus'}
ANI = {'zebra', 'sheep', 'dog', 'cat', 'cow', 'giraffe', 'horse', 'bear', 'elephant'}


def to_cls(l):
    return "person" if l == "person" else "vehicle" if l in VEH else "animal" if l in ANI else None


def iou_matrix(a, b):
    """a: (n,4) xyxy, b: (m,4) xyxy -> (n,m)"""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)


# ---------------------------------------------------------------- detections

def load_dets(path, thr_low=0.1, videos=None):
    d = pd.read_csv(path, usecols=["video_id", "frame", "x1", "y1", "x2", "y2", "score", "label"], dtype={"video_id": str})
    d = d[d.score >= thr_low].copy()
    if videos is not None:
        d = d[d.video_id.isin(videos)]
    d["cls"] = d.label.map(to_cls)
    return d[d.cls.notna()].drop(columns="label")


def merge_dets(*parts, iou=0.5):
    """union of several detector runs: same-class NMS per (video, frame, class), best score first."""
    if len(parts) == 1:
        return parts[0]
    d = pd.concat(parts, ignore_index=True)
    d = d.sort_values(["video_id", "frame", "cls", "score"], ascending=[True, True, True, False]).reset_index(drop=True)
    key = d.video_id.astype(str) + "|" + d.frame.astype(str) + "|" + d.cls
    starts = np.flatnonzero(np.r_[True, key.to_numpy()[1:] != key.to_numpy()[:-1]])
    ends = np.r_[starts[1:], len(d)]
    B = d[["x1", "y1", "x2", "y2"]].to_numpy()
    keep = np.zeros(len(d), bool)
    for s, e in zip(starts, ends):
        if e - s == 1:
            keep[s] = True; continue
        alive = np.ones(e - s, bool); b = B[s:e]
        for i in range(e - s):
            if not alive[i]:
                continue
            keep[s + i] = True
            if i + 1 < e - s:
                alive[i + 1:] &= iou_matrix(b[i:i + 1], b[i + 1:])[0] < iou
    return d[keep]


# ---------------------------------------------------------------- tracker

def group_nms(boxes, scores, cls, thr=0.7):
    """NMS inside each class group (all vehicle labels are one group). returns kept indices."""
    keep = []
    for c in np.unique(cls):
        idx = np.where(cls == c)[0]
        idx = idx[np.argsort(-scores[idx])]
        while len(idx):
            i = idx[0]; keep.append(i)
            if len(idx) == 1:
                break
            iou = iou_matrix(boxes[i:i + 1], boxes[idx[1:]])[0]
            idx = idx[1:][iou < thr]
    return np.array(sorted(keep), dtype=int)


_F = np.eye(8); _F[:4, 4:] = np.eye(4)          # constant velocity, dt = 1 frame
_H = np.eye(4, 8)
W_POS, W_VEL = 1 / 20, 1 / 160                  # DeepSORT defaults


def xyxy_to_z(b):
    w, h = b[..., 2] - b[..., 0], b[..., 3] - b[..., 1]
    return np.stack([b[..., 0] + w / 2, b[..., 1] + h / 2, w / np.maximum(h, 1e-6), h], -1)


def x_to_xyxy(x):
    cx, cy, a, h = x[..., 0], x[..., 1], x[..., 2], np.maximum(x[..., 3], 1e-3)
    w = a * h
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], -1)


def kf_init(z):
    """z: (n,4) -> mean (n,8), cov (n,8,8)"""
    n = len(z); h = z[:, 3]
    mean = np.concatenate([z, np.zeros((n, 4))], 1)
    std = np.stack([2 * W_POS * h, 2 * W_POS * h, np.full(n, 1e-2), 2 * W_POS * h,
                    10 * W_VEL * h, 10 * W_VEL * h, np.full(n, 1e-5), 10 * W_VEL * h], 1)
    cov = np.zeros((n, 8, 8)); idx = np.arange(8)
    cov[:, idx, idx] = std ** 2
    return mean, cov


def kf_predict(mean, cov):
    h = mean[:, 3]; n = len(mean)
    std = np.stack([W_POS * h, W_POS * h, np.full(n, 1e-2), W_POS * h,
                    W_VEL * h, W_VEL * h, np.full(n, 1e-5), W_VEL * h], 1)
    Q = np.zeros((n, 8, 8)); idx = np.arange(8); Q[:, idx, idx] = std ** 2
    mean = mean @ _F.T
    cov = _F @ cov @ _F.T + Q
    return mean, cov


def kf_update(mean, cov, z):
    h = mean[:, 3]; n = len(mean)
    std = np.stack([W_POS * h, W_POS * h, np.full(n, 1e-1), W_POS * h], 1)
    R = np.zeros((n, 4, 4)); idx = np.arange(4); R[:, idx, idx] = std ** 2
    S = _H @ cov @ _H.T + R                                   # (n,4,4)
    PHt = cov @ _H.T                                          # (n,8,4)
    K = np.linalg.solve(S, PHt.transpose(0, 2, 1)).transpose(0, 2, 1)   # (n,8,4)  Kalman gain
    y = z - mean[:, :4]                                       # innovation (measurement - prediction)
    mean = mean + (K @ y[..., None])[..., 0]
    cov = cov - K @ _H @ cov
    return mean, cov


def match(pred_boxes, pred_cls, det_boxes, det_cls, iou_gate, dist_gate, dist_w=0.3):
    """Hungarian matching. cost = 1 - IoU + dist_w * (center distance / height). returns (pairs, un_t, un_d)"""
    nt, nd = len(pred_boxes), len(det_boxes)
    if nt == 0 or nd == 0:
        return [], list(range(nt)), list(range(nd))
    iou = iou_matrix(pred_boxes, det_boxes)
    pc = (pred_boxes[:, None, :2] + pred_boxes[:, None, 2:]) / 2
    dc = (det_boxes[None, :, :2] + det_boxes[None, :, 2:]) / 2
    ph = np.maximum(pred_boxes[:, 3] - pred_boxes[:, 1], 1)[:, None]
    dh = np.maximum(det_boxes[:, 3] - det_boxes[:, 1], 1)[None, :]
    dn = np.linalg.norm(pc - dc, axis=2) / np.maximum(ph, dh)
    ok = ((iou >= iou_gate) | (dn < dist_gate)) & (pred_cls[:, None] == det_cls[None, :])
    cost = 1 - iou + dist_w * np.minimum(dn, 2)
    cost[~ok] = 1e6
    r, c = linear_sum_assignment(cost)
    pairs = [(i, j) for i, j in zip(r, c) if ok[i, j]]
    mt = {i for i, _ in pairs}; md = {j for _, j in pairs}
    return pairs, [i for i in range(nt) if i not in mt], [j for j in range(nd) if j not in md]


DEFAULT = {'thr_high': 0.4, 'thr_low': 0.1, 'thr_new': 0.4, 'nms_iou': 0.7, 'n_init': 3, 'max_age': 20, 'iou_gate': 0.1, 'dist_gate': 0.6, 'low_iou_gate': 0.3}


def track_video(gv, **kw):
    """gv: detections of one video (frame, x1..y2, score, cls). returns DataFrame of track observations:
    tid, frame, x1, y1, x2, y2 (raw detector box), kx1..ky2 (Kalman-smoothed box), score, cls."""
    p = {**DEFAULT, **kw}
    mean = np.zeros((0, 8)); cov = np.zeros((0, 8, 8))
    tid = np.zeros(0, int); tcls = np.zeros(0, object); hits = np.zeros(0, int)
    since = np.zeros(0, int); conf = np.zeros(0, bool)
    next_id = 0; rows = []
    frames = gv.frame.to_numpy()
    order = np.argsort(frames, kind="stable"); gv = gv.iloc[order]
    B = gv[["x1", "y1", "x2", "y2"]].to_numpy(float); Sc = gv.score.to_numpy(); C = gv.cls.to_numpy()
    F = gv.frame.to_numpy()
    bounds = np.flatnonzero(np.diff(F)) + 1
    starts = np.r_[0, bounds]; ends = np.r_[bounds, len(F)]
    last_f = None
    for s, e in zip(starts, ends):
        f = F[s]
        # predict (one step per elapsed frame-step; frames are every 1 at 10 fps)
        steps = 1 if last_f is None else max(1, int(round(f - last_f)))
        for _ in range(steps):
            if len(mean):
                mean, cov = kf_predict(mean, cov)
        since += steps if last_f is not None else 0
        last_f = f
        b, sc, c = B[s:e], Sc[s:e], C[s:e]
        k = group_nms(b, sc, c, p["nms_iou"]); b, sc, c = b[k], sc[k], c[k]
        hi = sc >= p["thr_high"]; lo = ~hi
        bh, sch, ch = b[hi], sc[hi], c[hi]
        bl, scl, cl = b[lo], sc[lo], c[lo]
        pb = x_to_xyxy(mean) if len(mean) else np.zeros((0, 4))
        # stage 1: all tracks vs high boxes (confirmed first priority is implicit in cost)
        pairs, un_t, un_d = match(pb, tcls, bh, ch, p["iou_gate"], p["dist_gate"])
        matched = [(t, bh[d], sch[d]) for t, d in pairs]
        # stage 2: confirmed leftover tracks vs low boxes, IoU only
        un_t = np.array(un_t, int)
        cand = un_t[conf[un_t]] if len(un_t) else un_t
        if len(cand) and len(bl):
            pairs2, _, _ = match(pb[cand], tcls[cand], bl, cl, p["low_iou_gate"], 0.0)
            matched += [(cand[t], bl[d], scl[d]) for t, d in pairs2]
        if matched:
            ti = np.array([m[0] for m in matched]); zb = np.array([m[1] for m in matched])
            mean[ti], cov[ti] = kf_update(mean[ti], cov[ti], xyxy_to_z(zb))
            hits[ti] += 1; since[ti] = 0
            conf[ti] |= hits[ti] >= p["n_init"]
            kb = x_to_xyxy(mean[ti])
            for (t, box, score), kbox in zip(matched, kb):
                rows.append((tid[t], f, *box, *kbox, score, tcls[t]))
        # new tracks from unmatched high boxes
        new = [d for d in un_d if sch[d] >= p["thr_new"]]
        if new:
            m2, c2 = kf_init(xyxy_to_z(bh[new]))
            mean = np.concatenate([mean, m2]); cov = np.concatenate([cov, c2])
            ids = np.arange(next_id, next_id + len(new)); next_id += len(new)
            tid = np.r_[tid, ids]; tcls = np.r_[tcls, ch[new]].astype(object)
            hits = np.r_[hits, np.ones(len(new), int)]; since = np.r_[since, np.zeros(len(new), int)]
            conf = np.r_[conf, np.zeros(len(new), bool) | (p["n_init"] <= 1)]
            for d, i in zip(new, ids):
                rows.append((i, f, *bh[d], *bh[d], sch[d], ch[d]))
        # life cycle: tentative tracks die at first miss, confirmed after max_age
        alive = np.where(conf, since <= p["max_age"], since == 0)
        mean, cov, tid, tcls, hits, since, conf = (a[alive] for a in (mean, cov, tid, tcls, hits, since, conf))
    out = pd.DataFrame(rows, columns=["tid", "frame", "x1", "y1", "x2", "y2", "kx1", "ky1", "kx2", "ky2", "score", "cls"])
    # drop tracks that were never confirmed
    n = out.groupby("tid").frame.transform("size")
    return out[n >= p["n_init"]].reset_index(drop=True)


def track_all(dets, **kw):
    res = []
    for vid, gv in dets.groupby("video_id", sort=False):
        t = track_video(gv, **kw); t.insert(0, "video_id", vid); res.append(t)
    return pd.concat(res, ignore_index=True)


# ---------------------------------------------------------------- ID-switch split

def split_tracks(tr, jump=0.8):
    tr = tr.sort_values(["video_id", "tid", "frame"]).reset_index(drop=True)
    cx = (tr.x1 + tr.x2) / 2; cy = (tr.y1 + tr.y2) / 2; h = (tr.y2 - tr.y1).clip(lower=1)
    same = (tr.video_id == tr.video_id.shift()) & (tr.tid == tr.tid.shift())
    df = (tr.frame - tr.frame.shift()).clip(lower=1)
    step = np.hypot(cx - cx.shift(), cy - cy.shift()) / np.maximum(h, h.shift()) / df
    cut = same & (step > jump)
    seg = (~same | cut).cumsum()
    tr["tid"] = seg.astype(int)                         # new unique ids (unique across videos too)
    n = tr.groupby("tid").frame.transform("size")
    return tr[n >= 3].reset_index(drop=True), int(cut.sum())


# ---------------------------------------------------------------- features

FPS = 10.0
FEATS = ['net_disp', 'max_disp', 'path', 'straight', 'straight2s', 'straight5s', 'speed1s', 'speed3s', 'foot_disp', 'edge_disp', 'age', 'fill', 'score_mean', 'h_px', 'h_rel', 'h_ratio', 'aspect', 'at_edge', 'edge_frac', 'cls_person', 'cls_vehicle', 'cls_animal', 'in_mov_veh']


def _back(f, n_frames):
    """index of the latest observation at least n_frames before each observation (or 0)."""
    j = np.searchsorted(f, f - n_frames, side="right") - 1
    return np.maximum(j, 0)


def track_features(g, W, H):
    g = g.sort_values("frame")
    f = g.frame.to_numpy().astype(float)
    cx = ((g.kx1 + g.kx2) / 2).to_numpy(); cy = ((g.ky1 + g.ky2) / 2).to_numpy()
    k = g[["kx1", "ky1", "kx2", "ky2"]].to_numpy()
    raw = g[["x1", "y1", "x2", "y2"]].to_numpy()
    h = np.maximum(k[:, 3] - k[:, 1], 1); w = np.maximum(k[:, 2] - k[:, 0], 1)
    n = len(f); idx = np.arange(n)
    hmed = pd.Series(h).expanding().median().to_numpy()
    # causal start position: median of the first min(i+1, 5) observations
    m0 = min(n, 5)
    def causal_med(a):           # median of the first min(i+1, 5) observations; constant after the 5th
        a = np.asarray(a); out = np.empty((n,) + a.shape[1:])
        for i in range(m0):
            out[i] = np.median(a[:i + 1], axis=0)
        out[m0:] = out[m0 - 1]
        return out
    ax, ay, axy, h0 = causal_med(cx), causal_med(cy), causal_med(k), causal_med(h)
    net = np.hypot(cx - ax, cy - ay) / hmed
    step = np.r_[0, np.hypot(np.diff(cx), np.diff(cy))]
    path = np.cumsum(step) / hmed
    cum = np.cumsum(step)

    def window_straight(nf):
        j = _back(f, nf)
        d = np.hypot(cx - cx[j], cy - cy[j]); p = cum - cum[j]
        return np.where(p > 1e-6, d / np.maximum(p, 1e-6), 0.0)

    def speed(nf):
        j = _back(f, nf)
        dt = np.maximum(f - f[j], 1) / FPS
        return np.hypot(cx - cx[j], cy - cy[j]) / hmed / dt

    foot = np.hypot(cx - ax, k[:, 3] - axy[:, 3]) / hmed
    touch = np.stack([raw[:, 0] < 3, raw[:, 1] < 3, raw[:, 2] > W - 3, raw[:, 3] > H - 3], 1)  # left, top, right, bottom
    ed = np.abs(k - axy) / hmed[:, None]                                  # displacement of x1, y1, x2, y2
    ed = np.where(touch, 0.0, ed).max(1)
    at_edge = touch.any(1).astype(float)
    out = pd.DataFrame(dict(
        video_id=g.video_id.to_numpy(), tid=g.tid.to_numpy(), frame=f.astype(int),
        cx=cx, cy=cy, x1=raw[:, 0], y1=raw[:, 1], x2=raw[:, 2], y2=raw[:, 3], cls=g.cls.to_numpy(),
        net_disp=net, max_disp=np.maximum.accumulate(net), path=path,
        straight=np.where(path > 1e-6, net / np.maximum(path, 1e-6), 0.0),
        straight2s=window_straight(20), straight5s=window_straight(50),
        speed1s=speed(10), speed3s=speed(30), foot_disp=foot, edge_disp=ed,
        age=(f - f[0]) / FPS, fill=(idx + 1) / (f - f[0] + 1),
        score_mean=g.score.expanding().mean().to_numpy(),
        h_px=hmed, h_rel=hmed / H, h_ratio=h / h0, aspect=pd.Series(w / h).expanding().median().to_numpy(),
        at_edge=at_edge, edge_frac=np.cumsum(at_edge) / (idx + 1)))
    for c in ("person", "vehicle", "animal"):
        out[f"cls_{c}"] = float(g.cls.iloc[0] == c)
    return out


def add_in_moving_vehicle(F, speed_thr=0.3):
    """for person rows: largest share of the person box inside a vehicle box that moves (speed1s > thr) at that frame."""
    F["in_mov_veh"] = 0.0
    veh = F[(F.cls == "vehicle") & (F.speed1s > speed_thr)]
    if len(veh) == 0:
        return F
    per = F[F.cls == "person"]
    vg = {k: g[["x1", "y1", "x2", "y2"]].to_numpy() for k, g in veh.groupby(["video_id", "frame"])}
    vals = np.zeros(len(per))
    pb = per[["x1", "y1", "x2", "y2"]].to_numpy(); keys = list(zip(per.video_id, per.frame))
    for i, key in enumerate(keys):
        b = vg.get(key)
        if b is None:
            continue
        p = pb[i]
        iw = np.clip(np.minimum(p[2], b[:, 2]) - np.maximum(p[0], b[:, 0]), 0, None)
        ih = np.clip(np.minimum(p[3], b[:, 3]) - np.maximum(p[1], b[:, 1]), 0, None)
        vals[i] = (iw * ih).max() / max((p[2] - p[0]) * (p[3] - p[1]), 1)
    F.loc[per.index, "in_mov_veh"] = vals
    return F


def build(tracks, videos):
    parts = []
    for vid, gv in tracks.groupby("video_id", sort=False):
        W, H = videos.loc[vid, ["width", "height"]]
        for _, g in gv.groupby("tid", sort=False):
            parts.append(track_features(g, W, H))
    return add_in_moving_vehicle(pd.concat(parts, ignore_index=True))


# ---------------------------------------------------------------- decision and submission

def format_alarms(alarms) -> str:
    return ";".join(f"{t:.2f} {x:.4f} {y:.4f} {c}" for t, x, y, c in alarms) or ";"


def rule_sub(F, videos, k=1.0, min_age=1.0, min_edge_h=0):
    """min_edge_h > 0: a box touching the frame border must be at least min_edge_h px tall to raise an alarm
    (tiny partial boxes at the border jitter and move "more than one box height" without going anywhere)."""
    disp = np.where(F.at_edge > 0, np.maximum(F.net_disp, F.edge_disp), F.net_disp)
    d = F.assign(disp=disp)
    edge_ok = (F.at_edge == 0) | (F.h_px >= min_edge_h)
    ok = d[(d.disp > k) & (d.age >= min_age) & edge_ok].sort_values(["video_id", "tid", "frame"]).groupby(["video_id", "tid"]).head(1)
    W = videos.loc[ok.video_id, "width"].to_numpy(); H = videos.loc[ok.video_id, "height"].to_numpy()
    ok = ok.assign(t=ok.frame / 10, x=(ok.x1 + ok.x2) / 2 / W, y=(ok.y1 + ok.y2) / 2 / H)
    al = {v: list(zip(g.t, g.x, g.y, g.cls)) for v, g in ok.groupby("video_id")}
    return pd.DataFrame({"video_id": list(videos.index), "alarms": [format_alarms(sorted(al.get(v, []))) for v in videos.index]})


def small_persons(d, max_h=40):
    """keep only person boxes shorter than max_h px (used for the 3x3-tile run: its vehicle boxes add false alarms)."""
    return d[(d.cls == "person") & ((d.y2 - d.y1) < max_h)]


def run(det_parts, videos, split=False):
    """det_parts: list of detection DataFrames (already class-mapped); videos: DataFrame indexed by video_id with
    width, height. split=True cuts tracks at ID switches (candidate A); False keeps them (candidate B)."""
    d = merge_dets(*det_parts)
    tr = track_all(d)
    if split:
        tr, _ = split_tracks(tr, 0.8)
    return rule_sub(build(tr, videos), videos)

In [6]:
# 6) Merge, track, decide, write the submission
coco1280 = load_dets(f"{OUT}/dets_test_coco1280.csv.gz", 0.1)
cocotile = load_dets(f"{OUT}/dets_test_cocotile.csv.gz", 0.1)
ft = load_dets(f"{OUT}/dets_test_ft1280.csv.gz", 0.1)
ft = ft[(ft.cls == "person") & (ft.score >= 0.25)]          # fine-tuned model: person boxes only, score >= 0.25
for n, d in (("coco1280", coco1280), ("cocotile", cocotile), ("ft persons", ft)):
    print(f"{n:10s} videos {d.video_id.nunique()} boxes {len(d)}")
F = build(track_all(merge_dets(coco1280, cocotile, ft)), TEST)
sub = rule_sub(F, TEST, min_edge_h=20)
assert len(sub) == len(TEST) and sub.video_id.is_unique
sub.to_csv(f"{OUT}/submission.csv", index=False)
print("submission.csv alarms:", int(sub.alarms.str.count("person|vehicle|animal").sum()))
print(pd.read_csv(f"{OUT}/submission.csv").head())

coco1280   videos 48 boxes 227596
cocotile   videos 48 boxes 374187
ft persons videos 48 boxes 62287
submission.csv alarms: 311
   video_id                                             alarms
0  10a86472  1.70 0.2930 0.9394 person;2.30 0.2928 0.9431 p...
1  19c04643  13.30 0.8313 0.5877 vehicle;19.60 0.8203 0.605...
2  20581d85  3.60 0.8969 0.3903 person;3.80 0.8991 0.3912 p...
3  255589cc  36.90 0.9234 0.3767 person;48.10 0.8609 0.4007...
4  25d91ca9  11.50 0.0424 0.5512 person;12.10 0.0564 0.5531...
